In [2]:
import polars as pl
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'train-*.parquet')
OUTPUT_PATH_DIR = PROJECT_ROOT / 'data' / 'processed'

OUTPUT_PATH_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = str(OUTPUT_PATH_DIR / 'dataset_bersih.parquet')

In [3]:
df = pl.scan_parquet(DATA_PATH).head(100000).collect()

In [4]:
print(f'Ukuran awal dataset: {df.height:,} baris x {df.width} kolom')
display(df.head())

Ukuran awal dataset: 100,000 baris x 15 kolom


id,news_title,news_source,news_author,news_hostname,news_date,news_text,news_tags,news_kabkot,news_intent,news_image,update_date,create_date,news_guid,news_raw_data
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""1924973""","""Dana Kegiatan Isbat Nikah Baka…","""https://radarutara.disway.id/m…","""Wahyudi""","""disway.id""","""2026-04-27 01:30:00+00:00""","""Dana Kegiatan Isbat Nikah Baka…","""['apbd perubahan 2026', 'diaju…","""Kabupaten Mukomuko""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 03:08:50.074774+00:…",null,"""CBMiswFBVV95cUxOaDhvMVNUZFRlel…","""{'id': None, 'url': 'https://r…"
"""1925951""","""Satu Orang Meninggal dari Insi…","""https://www.ayokaltim.com/satu…","""Iswanto""","""ayokaltim.com""","""2026-04-27 03:19:46+00:00""","""Satu Orang Meninggal dari Insi…","""['Peristiwa di Samarinda;Tabra…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 05:26:20.891514+00:…",null,"""CBMimgFBVV95cUxQSkNndGh1VGxOek…","""{'id': None, 'url': 'https://w…"
"""1925952""","""DPRD Bongkar Ketidaksesuaian L…","""https://infosatu.co/dprd-bongk…","""Emmy Haryanti""","""infosatu.co""","""2026-04-27 03:59:29+00:00""","""Samarinda, infosatu.co – Klaim…","""['Abdul rohim;DPRD Samarinda;L…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 05:26:20.891632+00:…",null,"""CBMirwFBVV95cUxOMFMxOGNOWlZtT1…","""{'id': None, 'url': 'https://i…"
"""1925953""","""DPRD Dorong Penanganan Banjir …","""https://infosatu.co/dprd-doron…","""Emmy Haryanti""","""infosatu.co""","""2026-04-27 01:43:24+00:00""","""Samarinda, infosatu.co – Dewan…","""['Banjir;Deni Hakim Anwar;DPRD…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 05:26:20.891650+00:…",null,"""CBMirwFBVV95cUxPMGdaZU9wVTR0ZG…","""{'id': None, 'url': 'https://i…"
"""1924974""","""Kepung Pasar Karang Nabire, Ma…","""https://papuatengah.tribunnews…","""Calvin Eluis Erari""","""tribunnews.com""","""2026-04-27 02:14:39+00:00""","""Demo di Nabire Kepung Pasar Ka…","""[]""","""Kabupaten Nabire""","""{'intent': 'lapor_informasi', …",null,"""2026-04-27 03:09:48.135779+00:…",null,"""CBMi1wFBVV95cUxNbHNySlhzMDFFV1…","""{'id': None, 'url': 'https://p…"


In [5]:
print("Missing Values Setiap Kolom")
display(df.null_count())

Missing Values Setiap Kolom


id,news_title,news_source,news_author,news_hostname,news_date,news_text,news_tags,news_kabkot,news_intent,news_image,update_date,create_date,news_guid,news_raw_data
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,1395,0,329,0,0,0,0,10535,0,10040,89960,89959


In [6]:
jumlah_duplikat = df.height - df.unique().height
print(f"Jumlah baris duplikat: {jumlah_duplikat:,}")

Jumlah baris duplikat: 0


In [7]:
df_clean = df.with_columns([
    pl.col("news_author").fill_null("Tidak Diketahui"),
    pl.col("news_image").fill_null("Tidak Ada Gambar"),
    
    pl.col("news_title").str.strip_chars(),
    pl.col("news_text").str.strip_chars(),
    pl.col("news_tags").str.strip_chars()
]).drop_nulls(subset=["news_date"])

print("Data Setelah Dibersihkan")
print(f'Ukuran dataset bersih: {df_clean.height:,} baris x {df_clean.width} kolom')
display(df_clean.null_count())

Data Setelah Dibersihkan
Ukuran dataset bersih: 99,671 baris x 15 kolom


id,news_title,news_source,news_author,news_hostname,news_date,news_text,news_tags,news_kabkot,news_intent,news_image,update_date,create_date,news_guid,news_raw_data
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,9711,89960,89959


In [8]:
df_clean.write_parquet(OUTPUT_PATH)

print(f"Data bersih disimpan di:\n{OUTPUT_PATH}")

Data bersih disimpan di:
/home/jovyan/work/data/processed/dataset_bersih.parquet
